# Non-unique box key detector

Walks through `teal_nonunique_box_key.NonUniqueBoxKeyDetector` on the
two fixture contracts in `tests/box_key/`:

- `vuln/`: `asset_params_get AssetName` flows directly into `box_create`
  and `box_put` keys → 2 violations expected.
- `safe_id_prefix/`: the same field is concatenated with the asset id
  before being used as the key → 0 violations expected (default
  `concat` blocks taint).

Then demonstrates the extension hooks (custom `Source`, custom
`FlowRule`) on top of the same fixture set.

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path

# Pin the Linux codeql binary (VS Code on WSL may pick the Windows one).
os.environ.setdefault("CODEQL", "/home/argi/tools/codeql/codeql")

HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent.parent))

import tealtools.ssa as teal_ssa
from tealtools.stacksim import StackSimulation
from tealtools.dataflow.nonunique_box_key import (
    NonUniqueBoxKeyDetector,
    Source, Sink, FlowRule,
    DEFAULT_SOURCES, DEFAULT_SINKS,
    Violation,
)
from tealtools.ssa import Const

FIXTURES = HERE.parent.parent / "tests" / "tealtools" / "box_key"

## Vulnerable contract

The detector should flag both `box_create` and `box_put` since the
asset name reaches each one through identity-preserving ops only
(assert + a stack shuffle).

In [2]:
prog_vuln = teal_ssa.SSAProgram(FIXTURES / "vuln" / "db")
det = NonUniqueBoxKeyDetector(prog_vuln)
violations = det.detect()
print(f"{len(violations)} violation(s)")
for v in violations:
    print(" ", v.pretty())

2 violation(s)
  asset_params_get AssetName@prog.teal:7  →  box_create@prog.teal:10  (key = phi(V#2@L7))
  asset_params_get AssetName@prog.teal:15  →  box_put@prog.teal:18  (key = phi(V#2@L15))


Stack simulation gives a per-line view of the data flow into
each sink — useful for explaining *why* a finding fires.

In [3]:
sim = StackSimulation(prog_vuln)
print(sim.render(file="prog.teal"))

L   5  txna ApplicationArgs 0      IN  []                       OUT [V#1@L5]
L   6  btoi                        IN  [V#1@L5]                 OUT [V#1@L6]
L   7  asset_params_get AssetName  IN  [V#1@L6]                 OUT [V#2@L7, V#1@L7]
L   8  assert                      IN  [V#2@L7, V#1@L7]         OUT [V#2@L7]
L   9  pushint 64                  IN  [phi(V#2@L7)]            OUT [phi(V#2@L7), V#1@L9]
L  10  box_create                  IN  [phi(V#2@L7), V#1@L9]    OUT [V#1@L10]
L  11  pop                         IN  [V#1@L10]                OUT []
L  13  txna ApplicationArgs 0      IN  []                       OUT [V#1@L13]
L  14  btoi                        IN  [V#1@L13]                OUT [V#1@L14]
L  15  asset_params_get AssetName  IN  [V#1@L14]                OUT [V#2@L15, V#1@L15]
L  16  assert                      IN  [V#2@L15, V#1@L15]       OUT [V#2@L15]
L  17  txna ApplicationArgs 1      IN  [phi(V#2@L15)]           OUT [phi(V#2@L15), V#1@L17]
L  18  box_put                  

`tainted_operands()` exposes the full propagation set, which
is the diagnostic for "why did/didn't this flag?" questions. Each
SSAVar / Phi here was reached forward from an `asset_params_get
AssetName` output through the default identity-preserving rules.

In [4]:
sorted(repr(o) for o in det.tainted_operands())

['V#2@L15', 'V#2@L7', 'phi(V#2@L15)', 'phi(V#2@L7)']

## Safe contract

The asset id is concatenated with the name before being used as the
key. `concat` is not in the default identity-preserving op set, so
taint stops at the concat — the result has the asset id mixed in and
is unique even when two ASAs share a name.

In [5]:
prog_safe = teal_ssa.SSAProgram(FIXTURES / "safe_id_prefix" / "db")
violations_safe = NonUniqueBoxKeyDetector(prog_safe).detect()
print(f"{len(violations_safe)} violation(s) — expected 0")

0 violation(s) — expected 0


In [6]:
print(StackSimulation(prog_safe).render(file="prog.teal"))

L   5  txna ApplicationArgs 0      IN  []                           OUT [V#1@L5]
L   6  btoi                        IN  [V#1@L5]                     OUT [V#1@L6]
L   7  dup                         IN  [V#1@L6]                     OUT [V#2@L7, V#1@L7]
L   8  itob                        IN  [V#2@L7, V#1@L7]             OUT [V#2@L7, V#1@L8]
L   9  swap                        IN  [V#2@L7, V#1@L8]             OUT [V#2@L9, V#1@L9]
L  10  asset_params_get AssetName  IN  [V#2@L9, V#1@L9]             OUT [V#2@L9, V#2@L10, V#1@L10]
L  11  assert                      IN  [V#2@L9, V#2@L10, V#1@L10]   OUT [V#2@L9, V#2@L10]
L  12  concat                      IN  [phi(V#2@L9), phi(V#2@L10)]  OUT [V#1@L12]
L  13  pushint 64                  IN  [V#1@L12]                    OUT [V#1@L12, V#1@L13]
L  14  box_create                  IN  [V#1@L12, V#1@L13]           OUT [V#1@L14]
L  15  pop                         IN  [V#1@L14]                    OUT []
L  17  pushint 1                   IN  []           

## Extension: more non-unique fields

Add `asset_params_get AssetUnitName` (also non-unique) without
touching the default config — append to `DEFAULT_SOURCES`.

In [7]:
extra_source = Source(
    name="asset_params_get AssetUnitName",
    matches=lambda a: (
        a.op == "asset_params_get"
        and a.immediates.strip().split()[:1] == ["AssetUnitName"]
    ),
    tainted_outputs=lambda a: [2],
)

det_more = NonUniqueBoxKeyDetector(
    prog_vuln,
    sources=[*DEFAULT_SOURCES, extra_source],
)
v_more = det_more.detect()
print(f"{len(v_more)} violation(s) on vuln fixture (still 2 — fixture only uses AssetName)")
for v in v_more:
    print(" ", v.pretty())

2 violation(s) on vuln fixture (still 2 — fixture only uses AssetName)
  asset_params_get AssetName@prog.teal:7  →  box_create@prog.teal:10  (key = phi(V#2@L7))
  asset_params_get AssetName@prog.teal:15  →  box_put@prog.teal:18  (key = phi(V#2@L15))


## Built-in propagation through entropy-preserving ops

The detector ships with `DEFAULT_RULES` covering ops that produce
their output deterministically from a single bytes input — collisions
on the source persist through the op:

- **Hashes** (`sha256`, `keccak256`, `sha512_256`, `sha3_256`) —
  same input bytes always yield the same hash.
- **Slice ops** (`extract`, `extract3`, `extract_uint{16,32,64}`,
  `substring`, `substring3`) — a slice of a non-unique field is at
  best as unique as the field itself.
- **`concat`** when every non-tainted operand is statically constant
  (a `Const` operand or an SSAVar with `const_value` set) — adding
  a constant prefix/suffix doesn't add per-asset entropy.

Three fixtures exercise each branch:

In [8]:
for case in ["vuln_concat_const", "vuln_hash", "vuln_extract"]:
    p = teal_ssa.SSAProgram(FIXTURES / case / "db")
    vs = NonUniqueBoxKeyDetector(p).detect()
    print(f"  {case}: {len(vs)} violation(s)")
    for v in vs:
        print(f"    {v.pretty()}")

  vuln_concat_const: 1 violation(s)
    asset_params_get AssetName@prog.teal:7  →  box_create@prog.teal:13  (key = V#1@L11)


  vuln_hash: 1 violation(s)
    asset_params_get AssetName@prog.teal:7  →  box_create@prog.teal:11  (key = V#1@L9)


  vuln_extract: 1 violation(s)
    asset_params_get AssetName@prog.teal:7  →  box_create@prog.teal:11  (key = V#1@L9)


Disabling the defaults — pass `default_rules=[]` — falls back
to the original "shuffles only" behaviour. The new fixtures become
silent because their entropy-preserving ops now BLOCK taint.

In [9]:
for case in ["vuln_concat_const", "vuln_hash", "vuln_extract"]:
    p = teal_ssa.SSAProgram(FIXTURES / case / "db")
    vs = NonUniqueBoxKeyDetector(p, default_rules=[]).detect()
    print(f"  {case} with defaults disabled: {len(vs)} violation(s)")

  vuln_concat_const with defaults disabled: 0 violation(s)


  vuln_hash with defaults disabled: 0 violation(s)


  vuln_extract with defaults disabled: 0 violation(s)


Subsetting works the same way. `default_rules=[HASH_PROPAGATION_RULE]`
keeps only hash propagation — `vuln_hash` flags, the others don't.

In [10]:
from tealtools.dataflow.nonunique_box_key import HASH_PROPAGATION_RULE

for case in ["vuln_concat_const", "vuln_hash", "vuln_extract"]:
    p = teal_ssa.SSAProgram(FIXTURES / case / "db")
    vs = NonUniqueBoxKeyDetector(p, default_rules=[HASH_PROPAGATION_RULE]).detect()
    print(f"  {case} with only hash rule: {len(vs)} violation(s)")

  vuln_concat_const with only hash rule: 0 violation(s)


  vuln_hash with only hash rule: 1 violation(s)


  vuln_extract with only hash rule: 0 violation(s)
